#### <u>Read data in python environment.</u>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('banking_complaints_2023.csv')

df.head()

,Complaint ID,Date Received,Banking Product,Department,Issue ID,Complaint Description,State,ZIP,Bank Response
0,CID76118977,1/1/2023,Checking or savings account,CASA,I_3510635,on XX/XX/XX22 I opened a safe balance account ...,California,92311,Closed with monetary relief
1,CID98703933,1/1/2023,"Credit reporting, credit repair services, or o...",Credit Reports,I_3798538,There is an item from Bank of ABC on my credit...,California,91344,Closed with explanation
2,CID52036665,1/1/2023,Checking or savings account,CASA,I_3648593,On XX/XX/XX22 I found out that my account was ...,New York,10466,Closed with monetary relief
3,CID62581335,1/1/2023,Credit card or prepaid card,Credit Cards,I_6999080,I've had a credit card for years with Bank of ...,California,92127,Closed with monetary relief
4,CID65731164,1/1/2023,Checking or savings account,CASA,I_3648593,This issue has to do with the way that Bank of...,New Jersey,7946,Closed with explanation


#### <u>Check if the variables have correct datatypes. Make changes wherever necessary.</u>

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7011 entries, 0 to 7010
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Complaint ID           7011 non-null   object
 1   Date Received          7011 non-null   object
 2   Banking Product        7011 non-null   object
 3   Department             7011 non-null   object
 4   Issue ID               7011 non-null   object
 5   Complaint Description  7011 non-null   object
 6   State                  6984 non-null   object
 7   ZIP                    6981 non-null   object
 8   Bank Response          7011 non-null   object
dtypes: object(9)
memory usage: 493.1+ KB


In [3]:
total_rows = len(df)
for col in df.columns:
    print("=====", col, "=====")
    num_unique = df[col].nunique()
    print(f"Unique values: {num_unique} | Total values: {total_rows}")
    cardinality = num_unique / total_rows * 100
    print(f"Percentage unique: {cardinality :.2f}%")
    if cardinality < 2:
        print("Examples:", df[col].unique()[:5])
    print()

===== Complaint ID =====
Unique values: 7011 | Total values: 7011
Percentage unique: 100.00%

===== Date Received =====
Unique values: 288 | Total values: 7011
Percentage unique: 4.11%

===== Banking Product =====
Unique values: 17 | Total values: 7011
Percentage unique: 0.24%
Examples: ['Checking or savings account'
 'Credit reporting, credit repair services, or other personal consumer reports'
 'Credit card or prepaid card' 'Debt collection'
 'Money transfer, virtual currency, or money service']

===== Department =====
Unique values: 7 | Total values: 7011
Percentage unique: 0.10%
Examples: ['CASA' 'Credit Reports' 'Credit Cards' 'Loans' 'Remittance']

===== Issue ID =====
Unique values: 130 | Total values: 7011
Percentage unique: 1.85%
Examples: ['I_3510635' 'I_3798538' 'I_3648593' 'I_6999080' 'I_2805744']

===== Complaint Description =====
Unique values: 6984 | Total values: 7011
Percentage unique: 99.61%

===== State =====
Unique values: 55 | Total values: 7011
Percentage unique: 

Some columns need to be changed:
- "Date Received" should be a date type
- "Zip" should be an integer

The rest should be converted either to a string or category type.

This breakdown of each column's unique values above shows us which columns would be better as a category column.
- "Banking Product" only has 17 unique values that are likely fixed categories
- "Department", "State", and "Bank Response" also have a small amount of unique values that are likely fixed categories

The rest of the columns:
- "Complaint ID"
- "Issue ID"
- "Complaint Description"

Would be better as string columns because new rows may have new values.

However, for this assignment, only `Date Received`, `Banking Product`, `Department`, and `Complaint Description` are needed, so I will drop the other ones.

In [4]:
df = df[["Date Received", "Banking Product", "Department", "Complaint Description"]]

df["Date Received"] = pd.to_datetime(df["Date Received"])
df["Complaint Description"] = df["Complaint Description"].astype("string[pyarrow]")
df["Banking Product"] = df["Banking Product"].astype("category")
df["Department"] = df["Department"].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7011 entries, 0 to 7010
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Date Received          7011 non-null   datetime64[ns]
 1   Banking Product        7011 non-null   category      
 2   Department             7011 non-null   category      
 3   Complaint Description  7011 non-null   string        
dtypes: category(2), datetime64[ns](1), string(1)
memory usage: 8.0 MB


The needed columns have been converted to better data types, and none of them are missing any values.

#### <u>Find the date range</u>

In [5]:
earliest, latest = df['Date Received'].min().date(), df['Date Received'].max().date()
print(f"Date range: {earliest} to {latest}")

total_days = (latest - earliest).days + 1
months, days = divmod(total_days, 30)

print(f"Span: {months} months and {days} days")

Date range: 2023-01-01 to 2023-10-21
Span: 9 months and 24 days


#### <u>Define a function named preprocessing that executes the following series of pre-processing steps in order:</u>
- Convert text to lowercase
- Remove numbers
- Remove stopwords
- Remove punctuation
- Apply lemmatization

In [6]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

try:
    nltk.data.find("corpora/stopwords.zip")
except LookupError:
    nltk.download("stopwords")

try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    nltk.download("punkt_tab")

try:
    nltk.data.find("corpora/wordnet.zip/wordnet/")
except LookupError:
    nltk.download("wordnet")

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    # Convert to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenize
    tokens = nltk.word_tokenize(text)

    # Remove stopwords
    tokens = list(filter(lambda token: token not in stop_words, tokens))

    # Keep only letters
    tokens = map(lambda token: re.sub(r'[^a-z]', '', token), tokens)

    # Drop any empty strings
    tokens = [token for token in tokens if token]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # Convert back to string
    return ' '.join(tokens)

#### <u>Clean the text under ‘Complaint Description’ using the above function</u>

In [7]:
df['Complaint Description'].head()
df['cleaned_complaints'] = df['Complaint Description'].apply(preprocessing)
df['cleaned_complaints'].head()


0    xxxxxx opened safe balance account online usin...
1    item bank abc credit report belong must remove...
2    xxxxxx found account frozen apparent reason we...
3    ve credit card year bank abc xxxxxxxx paid bal...
4    issue way bank abc account linking bill pay pa...
Name: cleaned_complaints, dtype: object

#### <u>Convert the pre-processed text into a matrix of TF-IDF features for downstream modelling.</u>

In [8]:
from sklearn.model_selection import train_test_split

X = df['cleaned_complaints']
y = df['Department']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [10]:
X_train_tfidf_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=X_train.index
)

X_train_tfidf_df.head(5)

,aaa,abc,abc able,abc account,abc advised,abc agent,abc allowed,abc already,abc also,abc amount,...,york xxxx,you,young,yr,yr old,zero,zero balance,zero fraud,zip,zip code
4772,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3118,0.0,0.199446,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3566,0.0,0.071001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2663,0.0,0.054790,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
176,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### <u> Consider the department as a target variable and build a classification model.</u>
In order to effectively manage the process, it is critically important to categorize the complaint and pass on to the concerned product department. 

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
results_df = pd.DataFrame(columns=["Algorithm", "Best CV F1", "Test Accuracy", "Test Precision", "Test Recall", "Test F1"])

#### Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1.0, 5.0, 10.0],
    'class_weight': [None, 'balanced'],
    'l1_ratio': [0],
    'solver': ['lbfgs'],
    'max_iter': [1000, 5000]
}

lr_grid_search = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='f1_weighted'
)

lr_grid_search.fit(X_train_tfidf, y_train)

best_model = lr_grid_search.best_estimator_

predictions = best_model.predict(X_test_tfidf)

results_df.loc[len(results_df)] = {
    "Algorithm": "LogisticRegression",
    "Best CV F1": lr_grid_search.best_score_,
    "Test Accuracy": accuracy_score(y_test, predictions),
    "Test Precision": precision_score(y_test, predictions, average="weighted"),
    "Test Recall": recall_score(y_test, predictions, average="weighted"),
    "Test F1": f1_score(y_test, predictions, average="weighted"),
}

print("Best estimator:\n", lr_grid_search.best_estimator_, '\n')
print("Classification report:\n", classification_report(y_test, predictions))
print("Confusion matrix:\n", confusion_matrix(y_test, predictions))

Best estimator:
 LogisticRegression(C=5.0, class_weight='balanced', l1_ratio=0, max_iter=1000,
                   random_state=42) 

Classification report:
                 precision    recall  f1-score   support

          CASA       0.73      0.78      0.76       497
  Credit Cards       0.78      0.73      0.75       483
Credit Reports       0.77      0.73      0.75       333
         Loans       0.76      0.80      0.78       333
      Mortgage       0.89      0.89      0.89       254
        Others       0.70      0.48      0.57        77
    Remittance       0.56      0.65      0.60       127

      accuracy                           0.76      2104
     macro avg       0.74      0.72      0.73      2104
  weighted avg       0.76      0.76      0.76      2104

Confusion matrix:
 [[387  35   8   6   3   4  54]
 [ 65 353  29  21   2   6   7]
 [ 19  37 243  28   5   1   0]
 [  5  18  28 266  14   2   0]
 [  0   4   5  14 227   2   2]
 [ 18   5   2   9   4  37   2]
 [ 34   2   1   5  

#### Linear Support Vector Classifier

In [13]:
from sklearn.svm import LinearSVC

param_grid = {
    'C': [0.1, 0.5, 1.0, 5.0, 10.0],
    'loss': ['hinge', 'squared_hinge'],
    'class_weight': [None, 'balanced'],
    'max_iter': [10_000, 50_000]
}

linear_svc_grid_search = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='f1_weighted'
)

linear_svc_grid_search.fit(X_train_tfidf, y_train)

best_model = linear_svc_grid_search.best_estimator_

predictions = best_model.predict(X_test_tfidf)

results_df.loc[len(results_df)] = {
    "Algorithm": "LinearSVC",
    "Best CV F1": linear_svc_grid_search.best_score_,
    "Test Accuracy": accuracy_score(y_test, predictions),
    "Test Precision": precision_score(y_test, predictions, average="weighted"),
    "Test Recall": recall_score(y_test, predictions, average="weighted"),
    "Test F1": f1_score(y_test, predictions, average="weighted"),
}

print("Best estimator:\n", linear_svc_grid_search.best_estimator_, '\n')
print("Classification report:\n", classification_report(y_test, predictions))
print("Confusion matrix:\n", confusion_matrix(y_test, predictions))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best estimator:
 LinearSVC(C=0.1, class_weight='balanced', max_iter=10000, random_state=42) 

Classification report:
                 precision    recall  f1-score   support

          CASA       0.74      0.78      0.76       497
  Credit Cards       0.79      0.73      0.75       483
Credit Reports       0.76      0.73      0.74       333
         Loans       0.79      0.80      0.80       333
      Mortgage       0.87      0.91      0.89       254
        Others       0.67      0.47      0.55        77
    Remittance       0.56      0.69      0.62       127

      accuracy                           0.76      2104
     macro avg       0.74      0.73      0.73      2104
  weighted avg       0.76      0.76      0.76      2104

Confusion matrix:
 [[388  33   8   7   5   4  52]
 [ 58 351  34  17   2   9  12]
 [ 23  36 242  22   9   1   0]
 [  6  16  28 266  15   2   0]
 [  0   2   6  11 232   2   1]
 [ 20   7   2   7   3  36   2]
 [ 31   2   0   5   2   0  87]]


#### SGD Classifier

In [14]:
from sklearn.linear_model import SGDClassifier

param_grid = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],
    'alpha': [1e-5, 1e-4, 1e-3],
    'penalty': ['l2', 'elasticnet'],
    'class_weight': [None, 'balanced']
}

sgd_grid_search = GridSearchCV(
    SGDClassifier(max_iter=2000, random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='f1_weighted'
)

sgd_grid_search.fit(X_train_tfidf, y_train)

best_model = sgd_grid_search.best_estimator_

predictions = best_model.predict(X_test_tfidf)

results_df.loc[len(results_df)] = {
    "Algorithm": "SGDClassifier",
    "Best CV F1": sgd_grid_search.best_score_,
    "Test Accuracy": accuracy_score(y_test, predictions),
    "Test Precision": precision_score(y_test, predictions, average="weighted"),
    "Test Recall": recall_score(y_test, predictions, average="weighted"),
    "Test F1": f1_score(y_test, predictions, average="weighted"),
}

print("Best estimator:\n", sgd_grid_search.best_estimator_, '\n')
print("Classification report:\n", classification_report(y_test, predictions))
print("Confusion matrix:\n", confusion_matrix(y_test, predictions))

Best estimator:
 SGDClassifier(alpha=0.001, class_weight='balanced', loss='modified_huber',
              max_iter=2000, random_state=42) 

Classification report:
                 precision    recall  f1-score   support

          CASA       0.74      0.78      0.76       497
  Credit Cards       0.78      0.75      0.76       483
Credit Reports       0.77      0.73      0.75       333
         Loans       0.78      0.80      0.79       333
      Mortgage       0.88      0.91      0.90       254
        Others       0.69      0.48      0.56        77
    Remittance       0.60      0.65      0.63       127

      accuracy                           0.77      2104
     macro avg       0.75      0.73      0.74      2104
  weighted avg       0.77      0.77      0.77      2104

Confusion matrix:
 [[390  37   8   8   4   4  46]
 [ 58 362  29  17   2   8   7]
 [ 19  37 243  25   8   1   0]
 [  6  18  26 268  13   2   0]
 [  0   3   5  13 231   1   1]
 [ 21   7   2   6   3  37   1]
 [ 34   2   

#### Model Comparison

In [15]:
results_df.sort_values("Test F1", ascending=False).reset_index(drop=True).round(2)

,Algorithm,Best CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,SGDClassifier,0.77,0.77,0.77,0.77,0.77
1,LinearSVC,0.77,0.76,0.76,0.76,0.76
2,LogisticRegression,0.77,0.76,0.76,0.76,0.76


All models had very similar scores, but SGDClassifier with the hyperparameters shown in its output had the highest Test F1 score, so this would be a good model to use.

#### <u>Use SentimentIntensityAnalyzer to predict sentiments from the complaints.</u>

In [16]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon")

In [17]:
sia = SentimentIntensityAnalyzer()

df['polarity_score'] = df['Complaint Description'].apply(lambda text: sia.polarity_scores(text)['compound'])

df['sentiment'] = pd.cut(
    df['polarity_score'], 
    bins=[-np.inf, -0.05, 0.05, np.inf], 
    labels=['Negative', 'Neutral', 'Positive']
)

df.head()

,Date Received,Banking Product,Department,Complaint Description,cleaned_complaints,polarity_score,sentiment
0,2023-01-01,Checking or savings account,CASA,on XX/XX/XX22 I opened a safe balance account ...,xxxxxx opened safe balance account online usin...,-0.9488,Negative
1,2023-01-01,"Credit reporting, credit repair services, or o...",Credit Reports,There is an item from Bank of ABC on my credit...,item bank abc credit report belong must remove...,-0.1207,Negative
2,2023-01-01,Checking or savings account,CASA,On XX/XX/XX22 I found out that my account was ...,xxxxxx found account frozen apparent reason we...,-0.5267,Negative
3,2023-01-01,Credit card or prepaid card,Credit Cards,I've had a credit card for years with Bank of ...,ve credit card year bank abc xxxxxxxx paid bal...,-0.9368,Negative
4,2023-01-01,Checking or savings account,CASA,This issue has to do with the way that Bank of...,issue way bank abc account linking bill pay pa...,0.9357,Positive


In [18]:
df.sentiment.value_counts()

sentiment
Negative    3841
Positive    2861
Neutral      309
Name: count, dtype: int64

In [19]:
sentiment_by_dept = pd.crosstab(
    df['Department'], 
    df['sentiment'], 
    normalize='index'
).sort_values(by='Negative', ascending=False).round(2)

display(sentiment_by_dept)

sentiment,Negative,Neutral,Positive
Department,,,
Remittance,0.68,0.04,0.28
CASA,0.67,0.06,0.28
Others,0.59,0.01,0.40
Loans,0.55,0.05,0.40
Mortgage,0.52,0.04,0.44
Credit Cards,0.50,0.02,0.48
Credit Reports,0.41,0.05,0.54


This shows the sentiment breakdown for each department and sorts by most negative at the top. It shows that the Remittance department has the most negative sentiment, followed by CASA and Others.

In [20]:
product_sentiment = (
    df.groupby('Banking Product', observed=True)
    .agg(
        group_size=('polarity_score', 'size'),
        mean_polarity=('polarity_score', 'mean'),
        percent_negative=('sentiment', lambda s: (s == 'Negative').mean() * 100)
    )
    .sort_values('group_size', ascending=False)
    .round(2)
)

display(product_sentiment)

,group_size,mean_polarity,percent_negative
Banking Product,,,
Checking or savings account,1655,-0.32,66.53
Credit card or prepaid card,1233,-0.08,51.74
Mortgage,848,-0.06,52.00
Debt collection,748,-0.21,60.70
"Credit reporting, credit repair services, or other personal consumer reports",559,0.06,41.50
Credit reporting,550,0.08,39.64
"Money transfer, virtual currency, or money service",411,-0.32,67.40
Credit card,359,0.15,41.23
Bank account or service,256,-0.14,58.98


This table breaks down several things for each banking product:

- `group_size` shows how many complaints are about that product. It is sorted from biggest to smallest groups.
- `mean_polarity` shows the average sentiment score for that product. 
- `percent_negative` shows the percentage of complaints that are negative.

We can see that the products that get the most complaints are `Checking or savings account`, followed by `Credit card or prepaid card` and `Mortgage`. This can help the bank decide where attention is most needed.

The table could also be sorted by average polarity score, and that would show that `Money transfers` has the lowest sentiment, followed by `Money transfer, virtual currency, or money service` and `Checking or savings account`.

#### <u>How can the score be used by the bank? Share your insights.</u>

- **Specific problem detection:** Complaints can be grouped by the product type, and TF-IDF can be used to extract common keywords to identify specific problems. Each department can then focus on the biggest weak points.

- **Severity detection**: The sentiment score and specific words or phrases could be used to automatically detect severity. For example, highly negative complaints or complaints that mention legal action or severe outcomes should get the highest severity. The model could also be used to route these to the right department.

- **Advanced models for better extraction:** More advanced models such as ones based on transformers could be applied to extract more detailed information such as specific aspects that customers complain about or praise, and ideas that they may have given for improvement or features.

Using these tools and ideas, the bank can focus on weak areas to improve their efficiency, products, and reputation.